In [1]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from scipy import sparse

In [5]:
model = joblib.load("logistic_model.pkl")

X_scraped_tfidf = joblib.load("../feature_engineering/scraped_tfidf_matrix.pkl")  # sparse matrix
X_scraped_struct = joblib.load("../feature_engineering/scraped_feature_list.pkl") # DataFrame 

tfidf = joblib.load("../feature_engineering/tfidf_vectorizer.pkl")
svd = joblib.load("../XGboost_model_folder/tfidf_svd.pkl")

#need to transform with same svd as tested on
X_scraped_tfidf_svd = svd.transform(X_scraped_tfidf)

In [6]:
def to_numeric_sparse(df):
    df = df.copy()
    bool_cols = df.select_dtypes(include="bool").columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(np.int8)
    return sparse.csr_matrix(df.values)

X_scraped_struct_sparse = to_numeric_sparse(X_scraped_struct)

# Convert SVD output to sparse (optional)
X_scraped_tfidf_sparse = sparse.csr_matrix(X_scraped_tfidf_svd)

# Combine SVD + structured features
X_scraped_combined = sparse.hstack([X_scraped_tfidf_sparse, X_scraped_struct_sparse])



In [7]:
y_pred = model.predict(X_scraped_combined)

In [8]:
fraud_rows = X_scraped_struct.iloc[y_pred == 1]  # rows predicted as 1
print(fraud_rows)


      salary_avg_norm  location_legitimacy  loc_len  loc_word_count  \
1        1.110914e-03                    5       14               3   
2        9.831600e-04                    5       48               9   
3        7.529442e-07                    5       49               7   
4        3.999368e-04                    5       11               2   
5        5.431196e-04                    1        9               1   
...               ...                  ...      ...             ...   
8222     1.641665e-06                    5       11               2   
8223     8.331759e-07                    3       23               3   
8224     8.665153e-04                    5       48               7   
8225     2.110712e-06                    5       17               3   
8226     1.033151e-03                    1       24               3   

      has_us_prefix  has_state_code  starts_with_direction  contains_digits  \
1                 0               1                      0          

In [9]:
print(len(X_scraped_struct.iloc[y_pred == 1]))
print(len(X_scraped_struct.iloc[y_pred == 0]))

8115
112
